In [ ]:
import json, re, os, tempfile
import glob, pandas as pd
from tqdm import tqdm
import numpy as np

In [24]:
dfs = [pd.read_parquet(f) for f in glob.glob("./dummy/*.parquet")]
keys = list(dfs[0].columns)
for key in keys:
    print(key)

author
title
content
source
domain
raw_url
publication_date
article_id


In [25]:
col_list = []
col = ""
while col != "0":
    col = input()
    if col == "0":
        continue
    if col not in keys:
        print(f"selected column ({col}) is not in keys")
        continue
    col_list.append(col)
print(col_list)

['title', 'content']


In [26]:
col_set = list(dict.fromkeys(col_list))
out = "out.jsonl"
counter = 0

def has_empty(flds):
    for fld in flds:
        if fld == "":
            return True
    return False

with open(out, "w", encoding="utf-8") as fo:
    for di, df in enumerate(dfs, 1):
        for values in tqdm(df[col_set].itertuples(index=False, name=None),
                          total=len(df), desc=f"DF {di} rows"):
            fields = list(values)
            counter += 1
            if counter < 5:
                print(fields)
            if has_empty(fields):
                continue
            fo.write(json.dumps(dict(zip(col_set, fields)), ensure_ascii=False) + "\n")

print(f"wrote JSON Lines to {out}")


DF 1 rows:   9%|▉         | 9224/103397 [00:00<00:02, 45246.90it/s]

['Elijah Cummings Just Asked 5 Questions The White House Is Scared To Death Of Answering', 'Rep. Elijah E. Cummings is demanding answers after presidential staff members reportedly passed confidential information to House Intelligence Committee Chairman Devin Nunes and helped him sneak onto White House grounds.\n\nCummings, who is the Ranking Member of the House Committee on Oversight and Government Reform, sent a letter to White House Counsel Donald McGahn and National Security Advisor Lt. Gen. H.R. McMaster on Friday in which he asked some questions the White House really doesn\'t want to answer.\n\nThe letter made it clear that Cummings wants to know just exactly who knew what when.\n\n"Over <copyright> <copyright> <copyright> <copyright> <copyright> <copyright> <copyright> have reported that staff who work directly for you contacted Rep. Devin Nunes, the Chairman of the House Permanent Select Committee on Intelligence, secured his entry into the White House complex on March 21, 201

DF 2 rows: 100%|██████████| 153810/153810 [00:03<00:00, 47611.84it/s]

wrote JSON Lines to out.jsonl


# split

In [27]:
ratio_train = 0.8
ratio_valid = 0.1
ratio_test = 0.1

In [28]:
jsonl_path = "out.jsonl"
chunksize = 10000

with open(jsonl_path, "rb") as f:
    total_lines = sum(1 for _ in f)
total_chunks = np.ceil(total_lines / chunksize) if total_lines else None

chunks = pd.read_json(jsonl_path, lines=True, chunksize=chunksize)

df = pd.concat(
    tqdm(chunks, total=total_chunks, desc="Reading JSONL", unit="chunk"),
    ignore_index=True
)

print(df.shape)

Reading JSONL: 100%|██████████| 26/26.0 [00:06<00:00,  3.88chunk/s]

(257207, 2)


In [29]:
n = len(df)
rng = np.random
idx = np.arange(n)
rng.shuffle(idx)

n_train = int(n * ratio_train)
n_valid = int(n * ratio_valid)
n_test = int(n * ratio_test)
remaining = n - (n_train + n_valid + n_test)
n_train += remaining

idx_train = idx[: n_train]
idx_valid = idx[n_train: n_train + n_valid]
idx_test = idx[n_train + n_valid: n_train + n_valid + n_test]

train_df = df.iloc[idx_train].reset_index(drop=True)
valid_df = df.iloc[idx_valid].reset_index(drop=True)
test_df = df.iloc[idx_test].reset_index(drop=True)

print(f"train: {train_df.shape}\nvalid: {valid_df.shape}\ntest: {test_df.shape}\n")

def write_jsonl(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in tqdm(df.itertuples(index=False, name=None),
                        total=len(df), desc=f"Writing {path}", unit="row"):
            rec = dict(zip(df.columns, row))
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

write_jsonl(train_df, "train.jsonl")
write_jsonl(valid_df, "valid.jsonl")
write_jsonl(test_df,  "test.jsonl")

train: (205767, 2)
valid: (25720, 2)
test: (25720, 2)



Writing test.jsonl: 100%|██████████| 25720/25720 [00:00<00:00, 43929.07row/s]


# ahh... cleanup?

In [30]:
slang_dict = {
    "fr":  "for real",
    "idk": "i do not know",
    "smh": "shaking my head",
    "lol": "laughing out loud",
}

SLANG_RE = re.compile(r"\b(" + "|".join(map(re.escape, slang_dict.keys())) + r")\b")

URL_RE = re.compile(r"http\S+", flags=0)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
NONALNUM_RE = re.compile(r"[^a-z0-9\s<>]")
WS_RE = re.compile(r"\s+")

In [31]:
def expand_slang(text: str):
    count = 0
    def repl(m):
        nonlocal count
        count += 1
        return slang_dict[m.group(1)]
    return SLANG_RE.sub(repl, text), count

In [32]:
def clean_text(text: str):
    original = text
    txt = text.lower()

    urls = len(URL_RE.findall(txt))
    txt = URL_RE.sub("<url>", txt)
    mentions = len(MENTION_RE.findall(txt))
    txt = MENTION_RE.sub("<user>", txt)
    hashtags = len(HASHTAG_RE.findall(txt))
    txt = HASHTAG_RE.sub(r"\1", txt)
    txt, slang = expand_slang(txt)
    nonalnum = len(NONALNUM_RE.findall(txt))
    txt = NONALNUM_RE.sub("", txt)
    ws_before = len(WS_RE.findall(txt))
    txt = WS_RE.sub(" ", txt).strip()
    ws_collapses = ws_before
    changed = int(txt != original)

    return txt, {
        "urls": urls,
        "mentions": mentions,
        "hashtags": hashtags,
        "slang": slang,
        "nonalnum": nonalnum,
        "ws_collapses": ws_collapses,
        "changed": changed,
    }

In [33]:
def summarize(tag, s):
    replaced = s["urls"] + s["mentions"] + s["hashtags"] + s["slang"]
    print(
        f"{tag}.jsonl: rows={s['rows']} | fields={s['fields']} | "
        f"rows_changed={s['rows_changed']} | fields_changed={s['fields_changed']} | "
        f"replaced={replaced} | removed={s['nonalnum']} | ws_collapses={s['ws_collapses']}"
    )

In [ ]:
def clean_jsonl_file(in_path, out_path=None, cols_to_clean=None):
    if out_path is None:
        out_path = os.path.splitext(in_path)[0] + ".cleaned.jsonl"

    with open(in_path, "rb") as f:
        total = sum(1 for _ in f)

    stats = {
        "rows": 0,
        "fields": 0,
        "fields_changed": 0,
        "rows_changed": 0,
        "urls": 0,
        "mentions": 0,
        "hashtags": 0,
        "slang": 0,
        "nonalnum": 0,
        "ws_collapses": 0,
    }

    dir_name = os.path.dirname(os.path.abspath(out_path)) or "."
    fd, tmp_path = tempfile.mkstemp(prefix=".tmp_", suffix=".jsonl", dir=dir_name)
    os.close(fd)

    try:
        with open(in_path, "r", encoding="utf-8") as fin, \
             open(tmp_path, "w", encoding="utf-8") as fout:

            for line in tqdm(fin, total=total, desc=f"Cleaning {os.path.basename(in_path)}", unit="row"):
                if not line.strip():
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    fout.write(line)
                    continue

                stats["rows"] += 1
                row_changed = False

                target_cols = (cols_to_clean if cols_to_clean is not None
                               else [k for k, v in rec.items() if isinstance(v, str)])

                for c in target_cols:
                    v = rec.get(c)
                    if not isinstance(v, str):
                        continue
                    cleaned, cnts = clean_text(v)
                    stats["fields"] += 1
                    if cnts["changed"]:
                        stats["fields_changed"] += 1
                        row_changed = True
                    for k in ("urls","mentions","hashtags","slang","nonalnum","ws_collapses"):
                        stats[k] += cnts[k]
                    rec[c] = cleaned

                if row_changed:
                    stats["rows_changed"] += 1

                fout.write(json.dumps(rec, ensure_ascii=False) + "\n")

        os.replace(tmp_path, out_path)

    except Exception:
        try: os.remove(tmp_path)
        except OSError: pass
        raise

    return stats

In [ ]:
train_stats = clean_jsonl_file("train.jsonl", out_path="train.jsonl")
valid_stats = clean_jsonl_file("valid.jsonl", out_path="valid.jsonl")
test_stats = clean_jsonl_file("test.jsonl",  out_path="test.jsonl")

Cleaning test.jsonl:   0%|          | 0/25720 [00:00<?, ?row/s]


In [ ]:
summarize("train", train_stats)
summarize("valid", valid_stats)
summarize("test",  test_stats)

train.jsonl: rows=0 | fields=0 | rows_changed=0 | fields_changed=0 | replaced=0 | removed=0 | ws_collapses=0
valid.jsonl: rows=0 | fields=0 | rows_changed=0 | fields_changed=0 | replaced=0 | removed=0 | ws_collapses=0
test.jsonl: rows=0 | fields=0 | rows_changed=0 | fields_changed=0 | replaced=0 | removed=0 | ws_collapses=0
